In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/counselling_system_Dataset.csv")

df.head()

,Year,Round,Institute,Branch,Category,Domicile,OpeningRank,ClosingRank,BranchSeats,TotalSeats,TotalCandidates
0,2014.0,R1,MITS,BT,UR/X/OP,Y,78121.0,80003.0,2.0,30.0,1356000.0
1,2014.0,R1,MITS,BT,OBC/X/OP,Y,122826.0,133005.0,3.0,30.0,1356000.0
2,2014.0,R1,MITS,BT,UR/X/OP,Y,99915.0,137259.0,8.0,30.0,1356000.0
3,2014.0,R1,MITS,BT,UR/X/F,Y,113348.0,179038.0,4.0,30.0,1356000.0
4,2014.0,R1,MITS,BT,UR/NCC/OP,Y,190302.0,190302.0,1.0,30.0,1356000.0


In [3]:
df.shape

(6070, 11)

In [5]:
df["OpeningPercentile"] = (
    1 - (df["OpeningRank"] / df["TotalCandidates"])
).clip(0, 1)

df["ClosingPercentile"] = (
    1 - (df["ClosingRank"] / df["TotalCandidates"])
).clip(0, 1)

In [6]:
df

,Year,Round,Institute,Branch,Category,Domicile,OpeningRank,ClosingRank,BranchSeats,TotalSeats,TotalCandidates,OpeningPercentile,ClosingPercentile
0,2014.0,R1,MITS,BT,UR/X/OP,Y,78121.0,80003.0,2.0,30.0,1356000.0,0.942389,0.941001
1,2014.0,R1,MITS,BT,OBC/X/OP,Y,122826.0,133005.0,3.0,30.0,1356000.0,0.909420,0.901914
2,2014.0,R1,MITS,BT,UR/X/OP,Y,99915.0,137259.0,8.0,30.0,1356000.0,0.926316,0.898777
3,2014.0,R1,MITS,BT,UR/X/F,Y,113348.0,179038.0,4.0,30.0,1356000.0,0.916410,0.867966
4,2014.0,R1,MITS,BT,UR/NCC/OP,Y,190302.0,190302.0,1.0,30.0,1356000.0,0.859659,0.859659
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6065,2025.0,INTERNAL,MITS,MECH,UR/X/F,Y,775426.0,775426.0,1.0,8.0,1475103.0,0.474324,0.474324
6066,2025.0,INTERNAL,MITS,MECH,UR/X/F,Y,1016946.0,1016946.0,1.0,8.0,1475103.0,0.310593,0.310593
6067,2025.0,INTERNAL,MITS,MECH,UR/X/OP,Y,484102.0,484102.0,1.0,8.0,1475103.0,0.671818,0.671818
6068,2025.0,INTERNAL,MITS,MECH,UR/X/OP,Y,488769.0,681736.0,3.0,8.0,1475103.0,0.668654,0.537838


In [7]:
training_rows = []

In [9]:
training_rows = []

for _, row in df.iterrows():

    # Skip rows with missing ranks
    if pd.isna(row["OpeningRank"]) or pd.isna(row["ClosingRank"]):
        continue

    opening_rank = int(row["OpeningRank"])
    closing_rank = int(row["ClosingRank"])

    branch = row["Branch"]
    category = row["Category"]
    round_name = row["Round"]
    domicile = row["Domicile"]

    total_candidates = row["TotalCandidates"]
    total_seats = row["TotalSeats"]
    branch_seats = row["BranchSeats"]

    year = row["Year"]

    # ------------------------------
    # POSITIVE SAMPLES
    # ------------------------------

    positive_ranks = np.linspace(
        opening_rank,
        closing_rank,
        8
    ).astype(int)

    for rank in positive_ranks:

        training_rows.append({

            "Year": year,
            "Rank": rank,
            "Branch": branch,
            "Category": category,
            "Round": round_name,
            "Domicile": domicile,

            "TotalCandidates": total_candidates,
            "TotalSeats": total_seats,
            "BranchSeats": branch_seats,

            "Label": 1
        })

    # ------------------------------
    # NEGATIVE SAMPLES
    # ------------------------------

    negative_ranks = np.linspace(
        closing_rank + 5000,
        closing_rank + 100000,
        5
    ).astype(int)

    for rank in negative_ranks:

        training_rows.append({

            "Year": year,
            "Rank": rank,
            "Branch": branch,
            "Category": category,
            "Round": round_name,
            "Domicile": domicile,

            "TotalCandidates": total_candidates,
            "TotalSeats": total_seats,
            "BranchSeats": branch_seats,

            "Label": 0
        })

In [11]:
ml_df = pd.DataFrame(training_rows)

ml_df.head(50)

,Year,Rank,Branch,Category,Round,Domicile,TotalCandidates,TotalSeats,BranchSeats,Label
0,2014.0,78121,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
1,2014.0,78389,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
2,2014.0,78658,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
3,2014.0,78927,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
4,2014.0,79196,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
5,2014.0,79465,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
6,2014.0,79734,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
7,2014.0,80003,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,1
8,2014.0,85003,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,0
9,2014.0,108753,BT,UR/X/OP,R1,Y,1356000.0,30.0,2.0,0


In [13]:
ml_df.tail(20)

,Year,Rank,Branch,Category,Round,Domicile,TotalCandidates,TotalSeats,BranchSeats,Label
75627,2025.0,654169,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,3.0,1
75628,2025.0,681736,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,3.0,1
75629,2025.0,686736,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,3.0,0
75630,2025.0,710486,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,3.0,0
75631,2025.0,734236,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,3.0,0
75632,2025.0,757986,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,3.0,0
75633,2025.0,781736,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,3.0,0
75634,2025.0,178,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,1.0,1
75635,2025.0,178,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,1.0,1
75636,2025.0,178,MECH,UR/X/OP,INTERNAL,Y,1475103.0,8.0,1.0,1


In [15]:
ml_df.shape

(75647, 11)

In [16]:
ml_df["RankPercentile"] = (
    1 - (
        ml_df["Rank"] /
        ml_df["TotalCandidates"]
    )
).clip(0, 1)

In [17]:
ml_df["SeatShare"] = (
    ml_df["BranchSeats"] /
    ml_df["TotalSeats"]
)

In [18]:
ml_df.to_csv(
    "../data/processed/ml_dataset.csv",
    index=False
)

print("✅ ML dataset generated successfully")

✅ ML dataset generated successfully
